### RAG Pipline - Data Ingestion to Vector DB Pipeline

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
from pathlib import Path

In [8]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"  Loaded{len(documents)} pages")


        except Exeption as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 7 PDF files to process

Processing: E-receipt.pdf
  Loaded2 pages

Processing: Explanatory Note Regarding Return from Leave of Absence.pdf
  Loaded1 pages

Processing: Explanatory Note_Dameli Kassym (3).pdf
  Loaded6 pages

Processing: IIPP_Final_Report_Dameli_Kassym.pdf
  Loaded5 pages

Processing: Itinerary.pdf
  Loaded3 pages

Processing: student_transcript (8).pdf
  Loaded2 pages

Processing: Trip.com.pdf
  Loaded2 pages

Total documents loaded: 21


In [10]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\E-receipt.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'E-receipt.pdf', 'file_type': 'pdf'}, page_content='Trip.com Travel Singapore Pte. Ltd.\nCompany No/GST Reg. No: 201613701E\nBooking No. 1688893809231567\nDate of Booking: 5:18 AM, October 4, 2025 (GMT+8)\nReceipt\nContact Info\nContact Name DAMELI KASSYM\nEmail dame*****ssym@nu.edu.kz\nPassenger & E-ticket No.\nDAMELI KASSYM 999-7379002837\nFlights\nTaipei - Beijing\nAir China CA190\nOctober 30, 2025 Economy class\nBeijing - Astana\nAir China CA791\nOctober 31, 2025 Economy class\nPrice Summary Amount\nFare\nTaxes & fees\nDelayed Baggage Protection\n125,916.00 KZT\n54,853.00 KZT\n2,735.00 KZT\nTotal (Visa 440043***06) 183,504.00 KZT\nThis receipt is automatically generated.\nTrip.com Travel Singapore Pte. Ltd.\n1/2'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'sou

In [13]:
### Text splitting

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """"Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

#Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]} ...")
        print(f"Metadata: {split_docs[0].metadata}")

        return split_docs

In [14]:
chunks = split_documents(all_pdf_documents)
chunks

Split 21 documents into 24 chunks

Example chunk:
Content: Trip.com Travel Singapore Pte. Ltd.
Company No/GST Reg. No: 201613701E
Booking No. 1688893809231567
Date of Booking: 5:18 AM, October 4, 2025 (GMT+8)
Receipt
Contact Info
Contact Name DAMELI KASSYM
Em ...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\E-receipt.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'E-receipt.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\E-receipt.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'E-receipt.pdf', 'file_type': 'pdf'}, page_content='Trip.com Travel Singapore Pte. Ltd.\nCompany No/GST Reg. No: 201613701E\nBooking No. 1688893809231567\nDate of Booking: 5:18 AM, October 4, 2025 (GMT+8)\nReceipt\nContact Info\nContact Name DAMELI KASSYM\nEmail dame*****ssym@nu.edu.kz\nPassenger & E-ticket No.\nDAMELI KASSYM 999-7379002837\nFlights\nTaipei - Beijing\nAir China CA190\nOctober 30, 2025 Economy class\nBeijing - Astana\nAir China CA791\nOctober 31, 2025 Economy class\nPrice Summary Amount\nFare\nTaxes & fees\nDelayed Baggage Protection\n125,916.00 KZT\n54,853.00 KZT\n2,735.00 KZT\nTotal (Visa 440043***06) 183,504.00 KZT\nThis receipt is automatically generated.\nTrip.com Travel Singapore Pte. Ltd.\n1/2'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'sou